# HGB walk-forward validation

Official B; one pooled model per lead, independent folds and 25-hour embargo. All 8 grid candidates × 4 folds × 9 leads ran. Ordered TRAIN: core, 14-day stopping tail, 14-day calibration tail, with 25-hour inner gaps. Stop by Brier every 10 trees, patience 30 and tolerance 1e-5; refit pre-calibration TRAIN at chosen iterations. Base rates fit only each actual fitting subset. Selected recency uses TRAIN-only weights. No random split, calendar/STL predictor, class resampling, xgboost or TEST evaluation.

Pooled raw VALID ECE >0.03 is the predeclared calibration trigger; coefficients would fit the untouched chronological TRAIN calibration tail. Candidate selection and conditional CIs use VALID, so they are not unbiased final scores.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
root = Path.cwd() if Path("src/airalerts").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))
from airalerts.evaluate import latest_results
out = root / "reports/model"
latest = latest_results(out / "results.csv")


In [2]:
hgb = latest.loc[latest.model.isin(["hgb", "hgb_platt"]) & latest.fold_id.eq("pooled") & latest.scope.eq("ALL")]
print(hgb[["model", "lead", "n", "brier", "bss_vs_clim", "bss_vs_persistence", "bss_vs_ses", "bss_ci_low", "bss_ci_high", "ece"]].to_string(index=False))
print(pd.read_csv(out / "hgb_selected.csv").to_string(index=False))


model  lead     n    brier  bss_vs_clim  bss_vs_persistence  bss_vs_ses  bss_ci_low  bss_ci_high      ece
  hgb     0 35136 0.054818     0.438384            0.042622    0.348814    0.408488     0.466447 0.011552
  hgb     1 35112 0.087572     0.102931            0.045656    0.048314    0.067421     0.140577 0.014253
  hgb     2 35088 0.091350     0.064397            0.044357    0.022056    0.034232     0.090652 0.019618
  hgb     3 35064 0.092005     0.057759            0.043288    0.013797    0.036896     0.077044 0.020856
  hgb     4 35040 0.092672     0.050946            0.035766   -0.001234    0.035472     0.067327 0.019745
  hgb     5 35016 0.093491     0.042314            0.028698   -0.008183    0.027880     0.057708 0.020721
  hgb     6 34992 0.094691     0.030049            0.019466   -0.019392    0.015248     0.044302 0.023574
  hgb    12 34848 0.094386     0.033725            0.030813   -0.011564    0.020035     0.045619 0.026458
  hgb    24 34560 0.093978     0.032412       

In [3]:
grid = pd.read_csv(out / "hgb_candidate_scores.csv")
grid["error_sum"] = grid.n * grid.brier
summary = grid.groupby(["lead", "learning_rate", "max_depth", "max_iter"])[["error_sum", "n"]].sum()
summary["pooled_brier"] = summary.error_sum / summary.n
print(summary[["pooled_brier"]].to_string())
print("Candidate-fold evaluations:", len(grid))
print(pd.read_csv(out / "hgb_train_tails.csv").to_string(index=False))


                                       pooled_brier
lead learning_rate max_depth max_iter              
0    0.05          3         100           0.055272
                             300           0.054854
                   5         100           0.054818
                             300           0.054824
     0.10          3         100           0.054884
                             300           0.054881
                   5         100           0.054828
                             300           0.054828
1    0.05          3         100           0.087582
                             300           0.087572
                   5         100           0.088098
                             300           0.088098
     0.10          3         100           0.087704
                             300           0.087704
                   5         100           0.088059
                             300           0.088059
2    0.05          3         100           0.091350
            

![HGB calibration](../reports/model/hgb_calibration.png)

HGB BSS versus climatology at lead 1 is 0.10293 and at lead 6 is 0.03005. Maximum pooled raw ECE is 0.02646, below 0.03, so real-data Platt was not triggered. At leads 12/24, BSS versus SES is -0.01156/-0.00791; this protocol provides no point-score superiority to SES there, and no FINAL inference is made.
